![BasQ banner](logo_cropped.png)

# <b>Simulating Quantum Dynamics: The Transverse-Field Ising Model </b>
**Author:** Benjamin Tirado  
**Created for:** BasQ Qiskit Fall Fest 2026 — Basque Quantum (BasQ)

<a id="goal"></a>
<div class="alert alert-block alert-success">
    
<b>Goal of this notebook: </b> Simulate the **real-time dynamics** of a small **transverse-field Ising model (TFIM)** spin chain on a quantum computer. We will build the time-evolution circuit using **first-order (Lie-Trotter) Trotterization**, run it on a noiseless simulator, and track the **total magnetization** of the chain as it evolves in time. By the end you will have a working template for digital quantum simulation that you can extend to other observables, larger systems, higher-order product formulas, and real hardware.
</div>

# <b>Table of Contents</b>

* [Background](#background)
* [Pre-requisites](#prereq)
* [Defining the Hamiltonian](#hamiltonian)
* [Trotterization: Turning $e^{-iHt}$ into Gates](#trotter)
* [Building the Time-Evolution Circuit](#circuit)
* [Measuring Magnetization over Time](#measure)
* [Results and Evaluation](#res)
* [Moving forward](#move)
* [Useful resources](#use)

## <b>Background </b> <a id="background"></a>

One of the most natural uses of a quantum computer is **simulating the dynamics of another quantum system** — Feynman's original motivation for building one. Given a system described by a Hamiltonian $\hat{H}$, quantum mechanics tells us that a state $|\psi(0)\rangle$ evolves in time according to the Schrödinger equation, whose solution is

$$
|\psi(t)\rangle = e^{-i \hat{H} t}\, |\psi(0)\rangle.
$$

The operator $U(t) = e^{-i\hat{H}t}$ is the **time-evolution operator**. If we could apply it directly on qubits, we could watch any quantum system evolve. The catch is that for a many-body Hamiltonian, $e^{-i\hat{H}t}$ is an exponentially large unitary and cannot in general be compiled into a short gate sequence exactly.

The standard workaround is **Trotterization** (a *product formula*). The Hamiltonian is usually a sum of simple terms, $\hat{H} = \sum_j \hat{H}_j$, each of which is easy to exponentiate on its own. Even though the terms do not commute, we can approximate the full evolution by chopping time into small slices and applying the easy pieces one after another. This turns an intractable exponential into a repeating pattern of elementary gates — exactly what a quantum computer runs.

As our testbed we use the **transverse-field Ising model (TFIM)**, a cornerstone of quantum magnetism. For a chain of $N$ spins it reads

$$
\hat{H} = -J \sum_{i=0}^{N-2} Z_i Z_{i+1} \; - \; h \sum_{i=0}^{N-1} X_i,
$$

where $Z_i$ and $X_i$ are Pauli operators on site $i$. The first term ($J$) is an **interaction** that favours neighbouring spins aligning along $z$; the second term ($h$) is a **transverse field** along $x$ that tries to flip them. The competition between these two non-commuting terms produces rich dynamics, and it is exactly this non-commutativity that makes Trotterization necessary — and interesting to study.

This kind of digital simulation of spin dynamics on real superconducting hardware was demonstrated by IBM Quantum and collaborators (for example Smith et al., *Simulating quantum many-body dynamics on a current digital quantum computer*, npj Quantum Information, 2019). Here we keep the system small so that every ingredient is easy to inspect and check against an exact classical calculation.

# <b>Pre-requisites </b> <a id="prereq"></a>
For starters, make sure you have installed the Qiskit SDK and its supporting modules: `qiskit_aer` (for noiseless simulations) and `qiskit_ibm_runtime` (for real-hardware experiments), as well as the visualization package `qiskit[visualization]`. In this notebook we build the time-evolution circuit directly from Qiskit's `PauliEvolutionGate` and the product-formula synthesis routines (`LieTrotter`, `SuzukiTrotter`), and estimate observables with the built-in `StatevectorEstimator`. We use `qiskit.quantum_info` to construct the Hamiltonian and to compute an exact reference for comparison.

In [ ]:
# %pip install qiskit qiskit_aer qiskit_ibm_runtime qiskit[visualization]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator

# <b>Defining the Hamiltonian </b> <a id="hamiltonian"></a>
We start by encoding the TFIM Hamiltonian as a `SparsePauliOp`: a weighted sum of Pauli strings, which is the representation Qiskit's evolution tools expect.

For a chain of $N$ qubits we build two groups of terms:

- **Interaction terms** $-J\, Z_i Z_{i+1}$ for each neighbouring pair $(i, i+1)$ along the chain (open boundary conditions, so no term wraps around from the last qubit to the first).
- **Field terms** $-h\, X_i$ for every qubit.

We take a chain of $N = 4$ spins with coupling $J = 1$ and field $h = 1$. This *critical-ish* regime, where interaction and field are comparable, gives non-trivial dynamics while staying small enough to verify exactly. The helper below constructs the operator for any $N$, $J$, $h$, so you can reuse it later.

In [ ]:
def tfim_hamiltonian(num_qubits, J=1.0, h=1.0):
    """Transverse-field Ising Hamiltonian on an open chain as a SparsePauliOp."""
    terms = []

    # Interaction terms: -J * Z_i Z_{i+1}
    for i in range(num_qubits - 1):
        pauli = ["I"] * num_qubits
        pauli[i] = "Z"
        pauli[i + 1] = "Z"
        # Qiskit reads Pauli strings right-to-left (qubit 0 is the rightmost char)
        terms.append(("".join(reversed(pauli)), -J))

    # Field terms: -h * X_i
    for i in range(num_qubits):
        pauli = ["I"] * num_qubits
        pauli[i] = "X"
        terms.append(("".join(reversed(pauli)), -h))

    return SparsePauliOp.from_list(terms)


num_qubits = 4
J, h = 1.0, 1.0
H = tfim_hamiltonian(num_qubits, J, h)

print(f"TFIM on {num_qubits} qubits (J={J}, h={h}):")
print(H)

# <b>Trotterization: Turning $e^{-iHt}$ into Gates </b> <a id="trotter"></a>
Our Hamiltonian splits into two groups of terms, $\hat{H} = \hat{H}_{ZZ} + \hat{H}_{X}$, that **do not commute** with each other. Because they don't commute, we cannot simply write $e^{-i\hat{H}t} = e^{-i\hat{H}_{ZZ}t}\, e^{-i\hat{H}_{X}t}$ — that equality is false.

The **first-order Lie-Trotter formula** makes it approximately true by taking small time steps. Splitting the total time $t$ into $n$ slices of size $\Delta t = t/n$,

$$
e^{-i\hat{H}t} \;\approx\; \left( e^{-i\hat{H}_{ZZ}\Delta t}\; e^{-i\hat{H}_{X}\Delta t} \right)^{n}.
$$

Each factor is now easy: $e^{-i\hat{H}_{X}\Delta t}$ is just a layer of single-qubit $R_X$ rotations, and $e^{-i\hat{H}_{ZZ}\Delta t}$ is a layer of two-qubit $ZZ$ rotations. Applying this pattern $n$ times realizes the full evolution.

The approximation is not free. For first-order Trotter, the error made in a single step scales as $\mathcal{O}(\Delta t^2)$, and the accumulated error over the whole evolution scales as $\mathcal{O}(\Delta t)$ — so **smaller steps mean higher accuracy but deeper circuits**. That accuracy-versus-depth tension is one of the central themes you will explore in the challenge. Higher-order product formulas (such as the second-order Suzuki-Trotter formula) reduce the error for the same step size at the cost of more gates per step; we import `SuzukiTrotter` here so you have it ready to compare.

Qiskit builds these circuits for us: a `PauliEvolutionGate` represents $e^{-i\hat{H}t}$, and the `synthesis` argument chooses the product formula (`LieTrotter` for first order). We just have to decide how many Trotter steps to use.

In [ ]:
# A single Trotter step for a small time dt, using the first-order Lie-Trotter formula.
dt = 0.2
single_step = PauliEvolutionGate(H, time=dt, synthesis=LieTrotter(reps=1))

demo = QuantumCircuit(num_qubits)
demo.append(single_step, range(num_qubits))
demo = demo.decompose(reps=2)  # unroll into elementary gates so we can see them

print(f"One Trotter step (dt={dt}):  depth = {demo.depth()}, "
      f"2-qubit gates = {demo.num_nonlocal_gates()}")
demo.draw("mpl", fold=-1)

# <b>Building the Time-Evolution Circuit </b> <a id="circuit"></a>
To follow the dynamics we need the state at many points in time, not just one. The cleanest way to do this is to fix the **step size** $\Delta t$ and build a circuit for each number of steps $n = 1, 2, 3, \dots$; the circuit with $n$ steps represents the state at time $t_n = n\,\Delta t$.

We also need a **starting state**. A common and physically meaningful choice is the fully polarized state $|00\dots0\rangle$ — every spin up along $z$. Under the TFIM, the transverse field immediately starts tipping the spins over, so the magnetization will begin to decay from its maximum value. That is the signal we will track.

The helper below prepares $|00\dots0\rangle$ and appends $n$ Trotter steps of size $\Delta t$.

In [ ]:
def trotter_evolution_circuit(H, num_qubits, dt, n_steps, order=1):
    """Prepare |00..0> and apply n_steps Trotter steps of size dt."""
    if order == 1:
        synth = LieTrotter(reps=1)
    else:
        synth = SuzukiTrotter(order=order, reps=1)

    qc = QuantumCircuit(num_qubits)
    # initial state |00..0> is the default; add an explicit barrier for clarity
    qc.barrier()

    step = PauliEvolutionGate(H, time=dt, synthesis=synth)
    for _ in range(n_steps):
        qc.append(step, range(num_qubits))

    return qc


# Example: 5 steps of size dt reaches time t = 5*dt
example = trotter_evolution_circuit(H, num_qubits, dt=0.2, n_steps=5)
print("Circuit for t =", 5 * 0.2)
example.draw("mpl", fold=-1)

# <b>Measuring Magnetization over Time </b> <a id="measure"></a>
The observable we track is the **total magnetization along $z$**, averaged per site:

$$
M_z(t) = \frac{1}{N}\sum_{i=0}^{N-1} \langle \psi(t) | Z_i | \psi(t)\rangle.
$$

At $t=0$ the state is $|00\dots0\rangle$, so every $\langle Z_i\rangle = +1$ and $M_z(0) = 1$. As the transverse field tips the spins, $M_z(t)$ decreases and oscillates. We build this observable as a `SparsePauliOp` and evaluate its expectation value with the `StatevectorEstimator`.

We sweep over increasing numbers of Trotter steps $n$, each giving the state at time $t_n = n\,\Delta t$, and record $M_z(t_n)$. To judge how good the Trotter approximation is, we compare against the **exact** evolution, obtained by exponentiating the Hamiltonian directly on the classical computer with `scipy` — feasible here only because the system is tiny.

In [ ]:
def magnetization_op(num_qubits):
    """(1/N) * sum_i Z_i as a SparsePauliOp."""
    terms = []
    for i in range(num_qubits):
        pauli = ["I"] * num_qubits
        pauli[i] = "Z"
        terms.append(("".join(reversed(pauli)), 1.0 / num_qubits))
    return SparsePauliOp.from_list(terms)


Mz = magnetization_op(num_qubits)
estimator = StatevectorEstimator()

# --- Trotterized evolution ---
dt = 0.2
max_steps = 30
times = np.array([n * dt for n in range(max_steps + 1)])

mz_trotter = []
for n in range(max_steps + 1):
    circ = trotter_evolution_circuit(H, num_qubits, dt=dt, n_steps=n)
    result = estimator.run([(circ, Mz)]).result()
    mz_trotter.append(float(result[0].data.evs))
mz_trotter = np.array(mz_trotter)

In [ ]:
# --- Exact reference evolution (classical, small systems only) ---
from scipy.linalg import expm

H_matrix = H.to_matrix()
Mz_matrix = Mz.to_matrix()
psi0 = Statevector.from_label("0" * num_qubits).data

mz_exact = []
for t in times:
    U = expm(-1j * H_matrix * t)
    psi_t = U @ psi0
    val = np.real(np.conj(psi_t) @ Mz_matrix @ psi_t)
    mz_exact.append(val)
mz_exact = np.array(mz_exact)

# <b>Results and Evaluation </b> <a id="res"></a>
We now plot the Trotterized magnetization against the exact result. Two features are worth watching:

- **Early times:** the Trotter curve should track the exact one closely — the accumulated error is still small.
- **Later times:** the curves gradually drift apart as Trotter error accumulates over many steps. Whether this drift is acceptable depends on the step size $\Delta t$ you chose.

This single plot already contains the seed of the whole challenge: change $\Delta t$ (or the number of steps to reach a given time) and the agreement changes. The lower panel shows the **error** $|M_z^{\text{Trotter}} - M_z^{\text{exact}}|$ over time, which is the quantity you will want to control.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(times, mz_exact, "k-", lw=2, label="Exact")
ax1.plot(times, mz_trotter, "o--", color="#1f77b4", ms=4,
         label=f"Trotter (1st order, dt={dt})")
ax1.set_ylabel(r"Magnetization  $M_z(t)$")
ax1.set_title(f"TFIM dynamics, N={num_qubits}, J={J}, h={h}")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(times, np.abs(mz_trotter - mz_exact), color="#d62728")
ax2.set_xlabel("Time  t")
ax2.set_ylabel("abs. error")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max error over the trajectory: {np.max(np.abs(mz_trotter - mz_exact)):.4f}")

# <b>Moving forward </b> <a id="move"></a>
The simulation in this notebook is intentionally minimal: a 4-spin chain, a single observable (magnetization), first-order Trotterization at one fixed step size, on a noiseless simulator. Each of those choices is a knob you can turn, and the challenge is about turning them thoughtfully. As before, the levels below tell you *what* to aim for, not *how* — the design decisions are yours.

### <b>Beginner — understand your simulation</b>
Deepen the analysis of the small-to-intermediate Ising chain from this notebook. There are several threads to pull on, and a strong submission investigates them together rather than in isolation:

- **Correlations.** Magnetization is the simplest observable, but the interesting physics of a spin chain lives in its **correlations**. Compute two-point correlation functions such as $\langle Z_i Z_j \rangle$ (and their connected version $\langle Z_i Z_j\rangle - \langle Z_i\rangle\langle Z_j\rangle$) as a function of distance and time. How does information spread along the chain?
- **Trotter step size.** Systematically study how the step size $\Delta t$ controls accuracy. For a fixed final time, how small does $\Delta t$ need to be before the Trotter result is trustworthy? Can you see the expected error scaling?
- **Accuracy vs. depth.** Smaller steps mean more accuracy but deeper circuits. Characterize this trade-off explicitly — plot error against circuit depth (or two-qubit gate count) and find the sweet spot.
- **Higher-order Trotter.** Swap the first-order Lie-Trotter formula for a **higher-order (Suzuki-Trotter) formula** and compare. For a given accuracy target, does the higher-order formula reach it with fewer total gates, or does the extra cost per step outweigh the benefit? (The `SuzukiTrotter` synthesis is already imported for you.)

### <b>Intermediate — recover physics the noise is trying to hide</b>
Push the simulation to a regime where **circuit depth becomes the bottleneck**: a longer chain, a two-dimensional-ish lattice, or longer evolution times, run under realistic noise (a noisy simulator or real hardware). At that point the raw signal degrades, and the challenge becomes extracting a meaningful observable anyway. Explore **error mitigation** (for example zero-noise extrapolation, measurement-error mitigation, or Pauli twirling) and **circuit-compression** or transpilation tricks that lower depth without changing the physics. Can you recover a magnetization or correlation curve that the unmitigated device gets wrong?

### <b>Advanced — simulate where classical checking gets hard</b>
Move toward a regime where a brute-force classical simulation becomes genuinely expensive — a larger system and/or a **kicked (Floquet) Ising** evolution — and use error mitigation to produce **trustworthy expectation values** on real hardware. The exact `expm` reference we used here will no longer be an option at scale, so part of the challenge is epistemic: how do you argue that your results are correct when you can't just diagonalize the Hamiltonian? Think about lightweight classical checks, limiting cases with known answers, and consistency tests across step sizes and mitigation settings.

### <b>Running on real hardware</b>
Whichever level you tackle, moving from the `StatevectorEstimator` to a real IBM Quantum device (such as `ibm_basquecountry`) changes the game. The heavy-hex connectivity means the $ZZ$ interactions in your Trotter step may not map onto physically connected qubits, forcing the transpiler to insert SWAPs that deepen the circuit. Mapping your chain onto a well-connected line of qubits, keeping Trotter steps shallow, and applying error suppression and mitigation all become part of designing a good simulation.

# <b>Useful resources </b> <a id="use"></a>

The following resources may be useful when extending this introductory simulation toward the full challenge.

### <b>Trotterization and time evolution in Qiskit </b>

- [Quantum real-time evolution using Trotterization](https://qiskit-community.github.io/qiskit-algorithms/tutorials/13_trotterQRTE.html)  
  Tutorial on simulating dynamics with product formulas, including Lie-Trotter and Suzuki-Trotter and circuit-depth analysis — the closest reference to this notebook.

- [`PauliEvolutionGate` documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.PauliEvolutionGate)  
  Reference for the gate representing $e^{-iHt}$ and how the product-formula synthesis is selected.

- [`LieTrotter` synthesis](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.synthesis.LieTrotter) and [`SuzukiTrotter` synthesis](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.synthesis.SuzukiTrotter)  
  Reference pages for the first-order and higher-order product formulas — directly relevant to the higher-order part of the beginner challenge.

- [`TrotterQRTE` documentation](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.TrotterQRTE.html)  
  A higher-level driver that automates Trotterized time evolution and can evaluate observables at every time step.

### <b>Observables, primitives, and operators </b>

- [`SparsePauliOp` documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp)  
  How to build Hamiltonians and observables as weighted sums of Pauli strings.

- [Estimator primitive documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.primitives.StatevectorEstimator)  
  Reference for computing expectation values, used here to measure magnetization.

- [Introduction to the primitives](https://quantum.cloud.ibm.com/docs/guides/primitives)  
  Guide to the Estimator and Sampler primitives and how they run on simulators and hardware.

### <b>Hardware execution, transpilation, and error mitigation </b>

- [Qiskit documentation](https://quantum.cloud.ibm.com/docs/guides/tools-intro)  
  General entry point for circuits, primitives, transpilation, and runtime execution on IBM Quantum.

- [Transpile with pass managers](https://quantum.cloud.ibm.com/docs/guides/transpile-with-pass-managers)  
  How to transpile circuits for a specific backend and its heavy-hex connectivity — relevant when mapping your chain onto hardware.

- [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-and-suppression-techniques)  
  Overview of the resilience options available through Qiskit Runtime — central to the intermediate and advanced challenges.

- [Combine error mitigation options with the Estimator primitive](https://quantum.cloud.ibm.com/docs/tutorials/combine-error-mitigation-techniques)  
  Tutorial on dynamical decoupling, measurement-error mitigation, gate twirling, and zero-noise extrapolation.

### <b>Background reading </b>

- [Smith, Kim, Pollmann & Knolle, *Simulating quantum many-body dynamics on a current digital quantum computer*, npj Quantum Information (2019)](https://www.nature.com/articles/s41534-019-0217-0)  
  Digital simulation of spin-chain dynamics on IBM hardware — the inspiration for the beginner track.

- [Kim et al., *Evidence for the utility of quantum computing before fault tolerance*, Nature (2023)](https://www.nature.com/articles/s41586-023-06096-3)  
  Large-scale kicked-Ising simulation with error mitigation — the inspiration for the advanced track.


# <b>Credits and license</b>

This notebook was written by **Benjamin Tirado** for the **BasQ Qiskit Fall Fest 2026**, organised by Basque Quantum (BasQ).

© 2026 Benjamin Tirado. The text, figures and explanations are released under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); the code is released under the
[Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0), the same license as Qiskit.

You are welcome to run, adapt and share this material — including as a starting point for your own
challenge submission — provided the attribution above is kept. If you reuse it publicly, please cite it as:

> B. Tirado, *Simulating Quantum Dynamics: The Transverse-Field Ising Model*, tutorial notebook, BasQ Qiskit Fall Fest, 2026.

Built with [Qiskit](https://www.ibm.com/quantum/qiskit), [NumPy](https://numpy.org/) and
[Matplotlib](https://matplotlib.org/), which remain the property of their respective authors and are
used under their own licenses.

*Questions, corrections or suggestions:* benjamin.tirado@ehu.eus